# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (files)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [ ]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Structured Streaming with Files",
                   master_url="spark://spark-master:7077")

su.spark

# Create a data stream from a local socket

### Connect Spark to the socket

In [ ]:
import pyspark.sql.functions as F
from pathlib import Path
import shutil

logs_schema = SparkUtils.generate_schema([("raw_line", "string")])

input_path = "/opt/spark/work-dir/data/streaming/logs/"

# Create the stream
logs_df = (su.spark.readStream
            .format("text")
            .option("maxFilesPerTrigger", 1) # Let's process one file at a time
            .schema(logs_schema)
            .load(input_path))

# Transform original dataframe
parsed_df = (
    logs_df
    .withColumn("parts",     F.split(F.col("raw_line"), r" \| "))
    .withColumn("timestamp", F.to_timestamp(F.col("parts")[0], "yyyy-MM-dd HH:mm:ss"))
    .withColumn("level",     F.trim(F.col("parts")[1]))
    .withColumn("message",   F.trim(F.col("parts")[2]))
    .withColumn("server",    F.trim(F.col("parts")[3]))
    .drop("parts", "raw_line")               # keep only the clean columns
    .filter(F.col("timestamp").isNotNull())  # skip malformed lines
)

# Let's create a summary
summary_df = (
    parsed_df
    .groupBy("server", "level")
    .count()
    .orderBy("server", "level")
)

# Clean checkpoint
checkpoint_path = "/opt/spark/work-dir/checkpoints/logs_checkpoint"
dir_path = Path(checkpoint_path)
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)

# Write stream in the destination
query_events = (
    parsed_df.writeStream
    .outputMode("append")        # append: show new rows only
    .format("console")
    .option("truncate", False)   # don't cut off long messages
    .option("numRows", 20)
    .option("checkpointLocation", checkpoint_path)
    .queryName("parsed_logs")
    .start()
)

query_summary = (
    summary_df.writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .queryName("summary_logs")
    .start()
)

print("   Press Ctrl+C to stop.\n")
su.spark.streams.awaitAnyTermination()


In [5]:
su.spark.stop()